In [85]:
base_dir = os.getcwd()
fasta_dir = os.path.join(base_dir, "fasta_files")
pop_dir = os.path.join(base_dir, "initial_population")

# Initial population based in BLOSUM62

In [86]:
import numpy as np
import pandas as pd
from Bio.Align import substitution_matrices
import os
import random

In [87]:
def punctuation_blosum(amn1, amn2):
    blosum62 = substitution_matrices.load("BLOSUM62")

    # Consultar el puntaje
    puntaje = blosum62[amn1, amn2]

    return puntaje

aminoacids = [
    'A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 
    'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V', 
    'B', 'Z', 'X', '*'
]

blosum = np.zeros((20,20))
for i in range(20):
    for j in range(20):
        blosum[i][j] = punctuation_blosum(aminoacids[i], aminoacids[j])

blosum = pd.DataFrame(blosum, index=aminoacids[:20], columns=aminoacids[:20])
print(blosum)

     A    R    N    D    C    Q    E    G    H    I    L    K    M    F    P  \
A  4.0 -1.0 -2.0 -2.0  0.0 -1.0 -1.0  0.0 -2.0 -1.0 -1.0 -1.0 -1.0 -2.0 -1.0   
R -1.0  5.0  0.0 -2.0 -3.0  1.0  0.0 -2.0  0.0 -3.0 -2.0  2.0 -1.0 -3.0 -2.0   
N -2.0  0.0  6.0  1.0 -3.0  0.0  0.0  0.0  1.0 -3.0 -3.0  0.0 -2.0 -3.0 -2.0   
D -2.0 -2.0  1.0  6.0 -3.0  0.0  2.0 -1.0 -1.0 -3.0 -4.0 -1.0 -3.0 -3.0 -1.0   
C  0.0 -3.0 -3.0 -3.0  9.0 -3.0 -4.0 -3.0 -3.0 -1.0 -1.0 -3.0 -1.0 -2.0 -3.0   
Q -1.0  1.0  0.0  0.0 -3.0  5.0  2.0 -2.0  0.0 -3.0 -2.0  1.0  0.0 -3.0 -1.0   
E -1.0  0.0  0.0  2.0 -4.0  2.0  5.0 -2.0  0.0 -3.0 -3.0  1.0 -2.0 -3.0 -1.0   
G  0.0 -2.0  0.0 -1.0 -3.0 -2.0 -2.0  6.0 -2.0 -4.0 -4.0 -2.0 -3.0 -3.0 -2.0   
H -2.0  0.0  1.0 -1.0 -3.0  0.0  0.0 -2.0  8.0 -3.0 -3.0 -1.0 -2.0 -1.0 -2.0   
I -1.0 -3.0 -3.0 -3.0 -1.0 -3.0 -3.0 -4.0 -3.0  4.0  2.0 -3.0  1.0  0.0 -3.0   
L -1.0 -2.0 -3.0 -4.0 -1.0 -2.0 -3.0 -4.0 -3.0  2.0  4.0 -2.0  2.0  0.0 -3.0   
K -1.0  2.0  0.0 -1.0 -3.0  1.0  1.0 -2.

In [88]:
def generate_new_sequence_blosum(sequence):
    new_sequence = ""
    for c in sequence:
        possibles_aminoacids =  blosum[blosum[c]>0][c] + 1
        total = sum(possibles_aminoacids)
        probs = possibles_aminoacids/total
        random_choosen_aminoacid = np.random.choice(probs.index, p=probs.values)
        new_sequence += random_choosen_aminoacid
    return new_sequence 

In [89]:
def get_new_sequences_blosum(sequence, n_population):
    new_sequences = []
    for i in range(n_population):
        new_seq = generate_new_sequence_blosum(sequence)
        new_sequences.append(new_seq)
    new_sequences = pd.DataFrame(new_sequences, columns=['Sequence'])
    new_sequences.insert(0, 'ID', [f'seq_{i+1}' for i in range(n_population)])
        
    return new_sequences

In [90]:
def generate_initial_population_by_blosum62(fasta_dir, pop_dir, population_size=200, seed=26):
    
    random.seed(seed)
    np.random.seed(seed)
    
    os.makedirs(pop_dir, exist_ok=True)
    
    fasta_files = [f for f in os.listdir(fasta_dir) if f.endswith(('.fasta', '.fa'))]
    
    if not fasta_files:
        print("  -> No FASTA files found for processing.")
        return

    for file in fasta_files:
        base_name = os.path.splitext(file)[0]
        fasta_path = os.path.join(fasta_dir, file)
        
        # Leer la secuencia original
        original_sequence = []
        with open(fasta_path, 'r') as f:
            for line in f:
                if not line.startswith('>'):
                    original_sequence.append(line.strip().upper())
        
        original_sequence = "".join(original_sequence)

        df_population = get_new_sequences_blosum(original_sequence, population_size)
        
        output_path = os.path.join(pop_dir, f"{base_name}.csv")
        df_population.to_csv(output_path, index=False)
        

# Initial population based in hydrophilic aminoacids

In [93]:
import os
import random
import csv

In [96]:
ALLOWED_MUTATIONS = {
    'A': ['A', 'C', 'T', 'V', 'I', 'L', 'M', 'F'], 
    'C': ['A', 'C', 'T', 'V', 'I', 'L', 'M', 'F'], 
    'D': ['D', 'E', 'G', 'H', 'K', 'N', 'P', 'Q', 'R', 'S', 'Y'], 
    'E': ['D', 'E', 'G', 'H', 'K', 'N', 'P', 'Q', 'R', 'S', 'W', 'Y'], 
    'F': ['A', 'C', 'V', 'I', 'L', 'M', 'F'], 
    'G': ['D', 'E', 'G', 'H', 'N', 'P', 'Q', 'R', 'S', 'W', 'Y'], 
    'H': ['D', 'E', 'G', 'H', 'K', 'N', 'P', 'Q', 'R', 'S', 'W', 'Y'], 
    'I': ['A', 'C', 'T', 'V', 'I', 'L', 'M', 'F'], 
    'K': ['D', 'E', 'H', 'K', 'N', 'P', 'Q', 'R', 'S', 'W', 'Y'], 
    'L': ['A', 'C', 'T', 'V', 'I', 'L', 'M', 'F'], 
    'M': ['A', 'C', 'T', 'V', 'I', 'L', 'M', 'F'], 
    'N': ['D', 'E', 'G', 'H', 'K', 'N', 'P', 'Q', 'R', 'S', 'W', 'Y'], 
    'P': ['D', 'E', 'G', 'H', 'K', 'N', 'P', 'Q', 'S', 'W', 'Y'], 
    'Q': ['D', 'E', 'H', 'K', 'N', 'P', 'Q', 'R', 'S', 'W', 'Y'], 
    'R': ['D', 'E', 'G', 'H', 'K', 'N', 'P', 'Q', 'R', 'S', 'W', 'Y'], 
    'S': ['D', 'E', 'G', 'H', 'K', 'N', 'P', 'Q', 'R', 'S', 'Y'], 
    'T': ['A', 'C', 'T', 'V', 'I', 'L', 'M'], 
    'V': ['A', 'C', 'T', 'V', 'I', 'L', 'M', 'F'], 
    'W': ['D', 'E', 'G', 'H', 'K', 'N', 'P', 'Q', 'R', 'W', 'Y'], 
    'Y': ['D', 'E', 'H', 'K', 'N', 'P', 'Q', 'R', 'S', 'W', 'Y']
}


In [105]:
def generate_initial_population_by_hydrophilic(fasta_dir, pop_dir, population_size=200, seed=26):

    random.seed(seed)
    os.makedirs(pop_dir, exist_ok=True)
    
    fasta_files = [f for f in os.listdir(fasta_dir) if f.endswith(('.fasta', '.fa'))]
    
    if not fasta_files:
        print("  -> No FASTA files found for processing.")
        return

    for file in fasta_files:
        base_name = os.path.splitext(file)[0]
        fasta_path = os.path.join(fasta_dir, file)
        
        # Read original sequence
        original_sequence = []
        with open(fasta_path, 'r') as f:
            for line in f:
                if not line.startswith('>'):
                    original_sequence.append(line.strip().upper())
        
        original_sequence = "".join(original_sequence)
        
        # Generate population
        population = []
        for _ in range(population_size):
            new_sequence = []
            
            for aa in original_sequence:
                if aa in ALLOWED_MUTATIONS:
                    # Choose a random letter from its predefined list in the dictionary
                    mutated_aa = random.choice(ALLOWED_MUTATIONS[aa])
                    new_sequence.append(mutated_aa)
                else:
                    # Keep the letter if it is a rare or unknown character (e.g., 'X')
                    new_sequence.append(aa)
                    
            population.append("".join(new_sequence))
            
        # Save to CSV
        output_path = os.path.join(pop_dir, f"{base_name}.csv")
        with open(output_path, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['ID', 'Sequence'])
            for i, seq in enumerate(population):
                writer.writerow([f'seq_{i+1}', seq])
        

# Generate initial population

In [102]:
generate_initial_population_by_blosum62(fasta_dir, pop_dir, population_size=50, seed=26)

In [106]:
generate_initial_population_by_hydrophilic(fasta_dir, pop_dir, population_size=200, seed=26)